<a href="https://colab.research.google.com/github/Arnavkhulbe/intelligent-document-qa-rag/blob/main/Intelligent_Document_QA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intelligent Document Question Answering System using RAG

### Generative AI Course Final Project

This project implements a Retrieval-Augmented Generation (RAG) system that allows users to ask questions about documents and receive context-aware answers using retrieved document information and a Large Language Model.

**Technologies:** Python, LangChain, ChromaDB, Hugging Face Embeddings, Groq LLM

In [1]:
!pip install -q langchain langchain-community langchain-openai langchain-chroma chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60

In [2]:
from google.colab import files

uploaded = files.upload()

Saving alice_in_wonderland.md to alice_in_wonderland.md


In [3]:
import os
import shutil

# Create the folder structure expected by the original project
os.makedirs("data/books", exist_ok=True)

# Move the uploaded document into the books folder
shutil.copy("alice_in_wonderland.md", "data/books/alice_in_wonderland.md")

print("Document ready!")

Document ready!


In [4]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "data/books",
    glob="*.md"
)

documents = loader.load()

print("Documents loaded:", len(documents))
print(documents[0].page_content[:500])

/tmp/ipykernel_1774/427091794.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader
ERROR:langchain_community.document_loaders.directory:Error loading file data/books/alice_in_wonderland.md


ImportError: unstructured package not found, please install it with `pip install unstructured`

In [5]:
!pip install -q "unstructured[md]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 41.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.2/169.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 14.0 MB/s eta 0:00:00


In [6]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "data/books",
    glob="*.md"
)

documents = loader.load()

print("Documents loaded:", len(documents))
print(documents[0].page_content[:500])

Documents loaded: 1
The Project Gutenberg eBook of Alice's Adventures in Wonderland

This ebook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this ebook or online at www.gutenberg.org. If you are not located in the United States, you will have to check the laws of the country where you are located before using this


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0].page_content[:500])

Number of chunks: 818

First chunk:

The Project Gutenberg eBook of Alice's Adventures in Wonderland


In [10]:
!pip install -q sentence-transformers langchain-huggingface

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("ChromaDB created successfully!")

ChromaDB created successfully!


In [20]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 8}
)

print("Retriever updated successfully!")

Retriever updated successfully!


In [21]:
query = "How does Alice meet the Mad Hatter?"

results = retriever.invoke(query)

print("Retrieved chunks:", len(results))

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:500])

Retrieved chunks: 8

--- Chunk 1 ---
“Really, now you ask me,” said Alice, very much confused, “I don’t think—”

“Then you shouldn’t talk,” said the Hatter.

--- Chunk 2 ---
Alice waited a little, half expecting to see it again, but it did not appear, and after a minute or two she walked on in the direction in which the March Hare was said to live. “I’ve seen hatters before,” she said to herself; “the March Hare will be much the most interesting, and perhaps as this is

--- Chunk 3 ---
“Have you guessed the riddle yet?” the Hatter said, turning to Alice again.

“No, I give it up,” Alice replied: “what’s the answer?”

“I haven’t the slightest idea,” said the Hatter.

“Nor I,” said the March Hare.

--- Chunk 4 ---
“Who’s making personal remarks now?” the Hatter asked triumphantly.

Alice did not quite know what to say to this: so she helped herself to some tea and bread-and-butter, and then turned to the Dormouse, and repeated her question. “Why did they live at the bottom of a well?”

-

In [15]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.6 MB/s eta 0:00:00


In [22]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [23]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("LLM loaded successfully!")

LLM loaded successfully!


In [24]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the following context.

Context:
{context}

Question:
{question}

If the answer is not present in the context, say:
"I don't know based on the provided document."
""")

question = "How does Alice meet the Mad Hatter?"

# Retrieve relevant chunks
retrieved_docs = retriever.invoke(question)

# Combine retrieved chunks
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

# Create the prompt
messages = prompt.format_messages(
    context=context,
    question=question
)

# Ask the LLM
response = llm.invoke(messages)

print("Answer:")
print(response.content)

Answer:
Alice meets the Hatter as she is walking in the direction in which the March Hare was said to live, but the context does not specify the exact moment or details of their initial meeting. The conversation between Alice and the Hatter starts abruptly, with the Hatter responding to Alice's confusion.
